### **Silver Layer**

In [0]:
# Schema conformance
from pyspark.sql.functions import col, to_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

# the drift batch
drift = (spark.table("workspace.bronze.readings_drift")
    .withColumnRenamed("ts", "read_timestamp")
    .withColumnRenamed("kwh", "consumption_kwh")
    .withColumn("read_timestamp", to_timestamp("read_timestamp", "dd/MM/yyyy HH:mm"))
    .select("reading_id", "meter_id", "read_timestamp", "consumption_kwh", "read_type"))

# the 32 normal files
main = (spark.table("workspace.bronze.readings")
    .withColumn("read_timestamp", to_timestamp("read_timestamp", "yyyy-MM-dd HH:mm:ss"))
    .select("reading_id", "meter_id", "read_timestamp", "consumption_kwh", "read_type"))

raw = main.unionByName(drift)
print(raw.count(), "rows after conforming + union")   # 84,577

In [0]:
typed   = raw.withColumn("consumption_kwh", col("consumption_kwh").cast("double"))
deduped = typed.dropDuplicates(["meter_id", "read_timestamp"])
print("removed", typed.count() - deduped.count(), "duplicate readings")   # 483

In [0]:
from pyspark.sql.functions import when, lit

valid_meters = (spark.table("workspace.bronze.meters")
                  .select("meter_id").distinct().withColumn("_ok", lit(True)))

checked = (deduped
    .join(valid_meters, "meter_id", "left")     # keeps every reading; marks which have a real parent meter
    .withColumn("reject_reason",
        when(col("_ok").isNull(),              lit("orphan_meter"))           # reading for a meter we don't have
        .when(col("consumption_kwh").isNull(), lit("null_consumption"))       # meter was offline
        .when(col("consumption_kwh") < 0,      lit("negative_consumption"))   # impossible value
        .otherwise(lit(None)))                                                # no reason = clean
    .drop("_ok"))

clean      = checked.filter(col("reject_reason").isNull()).drop("reject_reason")
quarantine = checked.filter(col("reject_reason").isNotNull())

In [0]:
clean = (clean
    .withColumn("is_estimated",   col("read_type") == "E")        # legitimate, but not a true measurement
    .withColumn("is_implausible", col("consumption_kwh") > 5.0))  # >5 kWh in 30 min for a home = suspicious spike

clean.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.readings")
quarantine.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.readings_quarantine")

print("clean:", spark.table("workspace.silver.readings").count())          # 83,631
print("quarantined:", spark.table("workspace.silver.readings_quarantine").count())  # 463

In [0]:
#Sanity check to confirm the cleaning is done right
import json
defects = json.load(open("/Volumes/workspace/brightwatt/raw/_dq_manifest.json"))["defects"]

q = spark.table("workspace.silver.readings_quarantine")
def caught(r): return q.filter(col("reject_reason") == r).count()

print("orphan_meter        :", caught("orphan_meter"),         "caught vs", defects["orphan_readings"]["count"], "injected")
print("null_consumption    :", caught("null_consumption"),     "caught vs", defects["null_consumption"]["count"], "injected")
print("negative_consumption:", caught("negative_consumption"), "caught vs", defects["negative_consumption"]["count"], "injected")

In [0]:
from pyspark.sql.functions import col, initcap, trim, lower, to_date

customers = (spark.table("workspace.bronze.customers")
    .withColumn("region", initcap(trim(col("region"))))   # 'NORTH WEST' / 'north west' -> 'North West'
    .withColumn("email", lower(trim(col("email"))))        # normalise the emails we do have
    .withColumn("has_email", col("email").isNotNull())     # contactability flag — never fabricate a missing email
    .withColumn("signup_date", to_date("signup_date")))    # text -> real date

customers.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.customers")

print("distinct regions:", customers.select("region").distinct().count())   # 6, down from 11
customers.groupBy("region").count().show()

In [0]:
from pyspark.sql.functions import col, to_date

meters = (spark.table("workspace.bronze.meters")
    .withColumn("install_date", to_date("install_date"))
    .withColumn("base_load_kw", col("base_load_kw").cast("double")))
    # supply_id and account_id stay STRINGS on purpose — they're identifiers, not numbers
meters.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.meters")

tariffs = (spark.table("workspace.bronze.tariffs")
    .withColumn("unit_rate_pence", col("unit_rate_pence").cast("double"))
    .withColumn("standing_charge_pence_per_day", col("standing_charge_pence_per_day").cast("double")))
tariffs.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.tariffs")

In [0]:
from pyspark.sql.functions import col, to_date, lower, trim

invoices = (spark.table("workspace.bronze.invoices")
    .withColumn("period_start", to_date("period_start"))
    .withColumn("period_end", to_date("period_end"))
    .withColumn("issued_date", to_date("issued_date"))
    .withColumn("total_kwh", col("total_kwh").cast("double"))
    .withColumn("amount_pence", col("amount_pence").cast("long"))
    .withColumn("status", lower(trim(col("status")))))     # 'Paid'/'PAID' -> 'paid' so filters group correctly
invoices.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.invoices")

payments = (spark.table("workspace.bronze.payments")
    .withColumn("paid_date", to_date("paid_date"))
    .withColumn("amount_pence", col("amount_pence").cast("long"))
    .withColumn("method", lower(trim(col("method")))))
payments.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.payments")

In [0]:
display(spark.sql("SHOW TABLES IN workspace.silver"))